# Lecture 19 · VAE on MNIST

Build a VAE · train on MNIST · visualize the latent space · interpolate between digits.

**Goal** · see the reparameterization trick in action and watch a model literally learn a smooth generative latent space.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)

## 1 · Data · MNIST

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
train_ds = datasets.MNIST('./data', train=True, download=True, transform=transform)
loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)

x_sample, y_sample = next(iter(loader))
print(f'batch shape: {x_sample.shape}   range: [{x_sample.min():.2f}, {x_sample.max():.2f}]')

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(x_sample[i, 0], cmap='gray')
    ax.set_title(y_sample[i].item())
    ax.axis('off')
plt.show()

## 2 · The VAE architecture

In [ ]:
class VAE(nn.Module):
    def __init__(self, d_in=784, d_h=400, d_z=2):
        super().__init__()
        self.enc1 = nn.Linear(d_in, d_h)
        self.enc_mu = nn.Linear(d_h, d_z)
        self.enc_logvar = nn.Linear(d_h, d_z)
        self.dec1 = nn.Linear(d_z, d_h)
        self.dec2 = nn.Linear(d_h, d_in)

    def encode(self, x):
        h = F.relu(self.enc1(x))
        return self.enc_mu(h), self.enc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std                # z = mu + sigma · eps

    def decode(self, z):
        h = F.relu(self.dec1(z))
        return torch.sigmoid(self.dec2(h))

    def forward(self, x):
        x = x.view(-1, 784)
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

def vae_loss(recon, x, mu, logvar, beta=1.0):
    # Reconstruction loss (BCE because output is Bernoulli)
    recon_loss = F.binary_cross_entropy(recon, x.view(-1, 784), reduction='sum')
    # KL divergence against N(0, I) in closed form
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl, recon_loss, kl

model = VAE(d_z=2).to(device)
print(f'params: {sum(p.numel() for p in model.parameters()):,}')

## 3 · Train · ~2 minutes on GPU

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
n_epochs = 10

for epoch in range(n_epochs):
    total_loss = 0
    total_recon = 0
    total_kl = 0
    for x, _ in loader:
        x = x.to(device)
        recon, mu, logvar = model(x)
        loss, recon_l, kl = vae_loss(recon, x, mu, logvar)
        opt.zero_grad(); loss.backward(); opt.step()
        total_loss += loss.item()
        total_recon += recon_l.item()
        total_kl += kl.item()
    n = len(loader.dataset)
    print(f'epoch {epoch:2d}  loss={total_loss/n:.2f}  recon={total_recon/n:.2f}  kl={total_kl/n:.2f}')

## 4 · Visualize the 2D latent space

Because we set `d_z = 2`, we can plot every training example as a point and colour by class.

In [ ]:
model.eval()
xs, ys = [], []
with torch.no_grad():
    for x, y in loader:
        mu, _ = model.encode(x.view(-1, 784).to(device))
        xs.append(mu.cpu())
        ys.append(y)
        if len(ys) * 128 > 5000: break
xs = torch.cat(xs).numpy()
ys = torch.cat(ys).numpy()

plt.figure(figsize=(7, 6))
scatter = plt.scatter(xs[:, 0], xs[:, 1], c=ys, cmap='tab10', s=3, alpha=0.6)
plt.colorbar(scatter, ticks=range(10), label='digit')
plt.xlabel('z₁'); plt.ylabel('z₂')
plt.title('VAE latent space · 2D projection')
plt.show()

## 5 · Generate · sample z ~ N(0, I) and decode

In [ ]:
with torch.no_grad():
    z = torch.randn(16, 2).to(device)
    gen = model.decode(z).view(-1, 28, 28).cpu()

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for ax, img in zip(axes.flat, gen):
    ax.imshow(img, cmap='gray')
    ax.axis('off')
plt.suptitle('random samples from N(0, I)')
plt.show()

## 6 · Interpolate · walk from one digit to another

In [ ]:
# Find a clear '7' and '2' in the dataset, encode them
x7 = x_sample[y_sample == 7][0:1].to(device)
x2 = x_sample[y_sample == 2][0:1].to(device)

with torch.no_grad():
    mu7, _ = model.encode(x7.view(-1, 784))
    mu2, _ = model.encode(x2.view(-1, 784))

    alphas = torch.linspace(0, 1, 10).to(device)
    zs = torch.stack([(1 - a) * mu7[0] + a * mu2[0] for a in alphas])
    decoded = model.decode(zs).view(-1, 28, 28).cpu()

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for ax, img, a in zip(axes, decoded, alphas):
    ax.imshow(img, cmap='gray')
    ax.set_title(f'α={a:.2f}')
    ax.axis('off')
plt.suptitle('Latent-space interpolation · 7 → 2')
plt.show()

**What you see** · every intermediate α produces a valid digit-like image. That is the guarantee the KL term bought us · the latent is smooth, continuous, and everywhere decodable.

## 7 · Try β-VAE

Re-train with `beta=4` — watch the latent tighten, samples get blurrier.

**Reflection**
- VAE training converges in minutes even on CPU (MNIST is tiny).
- 2D latent space is fine for visualization; real VAEs use 64+ dims.
- Samples look blurry because Bernoulli/MSE losses don't capture textures. GANs and diffusion fix this.